In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# 필요 라이브러리 임포트
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.preprocessing import StandardScaler
import numpy as np

# 1. 데이터 로드
df = pd.read_csv("/content/drive/MyDrive/AI+X/AI+X선도인재양성프로젝트(중급)/자율 프로젝트/그외 다른 아이디어/cotter_dataset.csv")

# 2. 입력(X): 사용자의 맛 프로파일
taste_features = ['Acidity', 'Sweet', 'Fruit', 'Nutty', 'Mouthfeel', 'Bitter', 'Sour']
X = df[taste_features]

# 3. 출력(y): 도징량, 추출량, 추출률
target_features = ['Dose', 'Brew Mass', 'Percent Extraction']
y = df[target_features]

# 4. 표준화
scaler_x = StandardScaler()
scaler_y = StandardScaler()
X_scaled = scaler_x.fit_transform(X)
y_scaled = scaler_y.fit_transform(y)

# 5. 학습/검증 데이터 분할
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_scaled, test_size=0.2, random_state=42)

# 6. 모델 정의 및 학습
model = MultiOutputRegressor(RandomForestRegressor(n_estimators=100, random_state=42))
model.fit(X_train, y_train)


MultiOutputRegressor(estimator=RandomForestRegressor(random_state=42))

In [4]:
def recommend_recipe(acidity, sweet, fruit, nutty, mouthfeel, bitter=0, sour=0, cup_size_ml=250):
    # 사용자 입력값 → DataFrame으로 변환
    user_input = pd.DataFrame([{
        'Acidity': acidity,
        'Sweet': sweet,
        'Fruit': fruit,
        'Nutty': nutty,
        'Mouthfeel': mouthfeel,
        'Bitter': bitter,
        'Sour': sour
    }])

    # 입력 스케일링 → 예측 → 역스케일링
    user_input_scaled = scaler_x.transform(user_input)
    pred_scaled = model.predict(user_input_scaled)
    pred = scaler_y.inverse_transform(pred_scaled)
    dose, brew_mass, extraction = [float(x) for x in pred[0]]

    # 컵 수 기준 계산
    cup_count = brew_mass / cup_size_ml
    dose_per_cup = dose / cup_count
    brew_per_cup = brew_mass / cup_count

    # 결과 반환
    return {
        f"1잔({cup_size_ml}ml) 기준 ": {
            "1잔당 도징량 (g)": round(dose_per_cup, 2),
            "1잔당 추출량 (g)": round(brew_per_cup, 2),
            "1잔당 추출률 (%)": round(extraction, 2)  # 추출률은 비율이라 동일
        }
    }


In [5]:
result = recommend_recipe(acidity=4, sweet=3, fruit=5, nutty=2, mouthfeel=3, cup_size_ml=250)
print(result)

{'1잔(250ml) 기준 ': {'1잔당 도징량 (g)': 19.42, '1잔당 추출량 (g)': 250.0, '1잔당 추출률 (%)': 18.91}}


## 딥러닝 기반

In [7]:
# =========================
# 0. 라이브러리
# =========================
import os
import numpy as np
import pandas as pd
from typing import Dict

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# 재현성
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# =========================
# 1. 데이터 로드
#  - 업로드 파일: /mnt/data/cotter_dataset.csv
#  - (Colab이면 본인 경로로 교체)
# =========================
DATA_PATH = "/content/drive/MyDrive/AI+X/AI+X선도인재양성프로젝트(중급)/자율 프로젝트/그외 다른 아이디어/cotter_dataset.csv"
assert os.path.exists(DATA_PATH), f"데이터가 없습니다: {DATA_PATH}"

df = pd.read_csv(DATA_PATH)

taste_features = ['Acidity', 'Sweet', 'Fruit', 'Nutty', 'Mouthfeel', 'Bitter', 'Sour']
target_features = ['Dose', 'Brew Mass', 'Percent Extraction']

X = df[taste_features].copy()
y = df[target_features].copy()

# 결측/이상치 간단 처리
X = X.replace([np.inf, -np.inf], np.nan).dropna()
y = y.loc[X.index].replace([np.inf, -np.inf], np.nan).dropna()
X = X.loc[y.index]

# =========================
# 2. 학습/검증/테스트 분할
# =========================
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.25, random_state=SEED)
X_val, X_test, y_val, y_test   = train_test_split(X_temp, y_temp, test_size=0.4, random_state=SEED)
# (train:val:test ≈ 75:15:10)

# =========================
# 3. 스케일러 (입/출력)
#   - NN은 입력/출력 스케일링이 수렴에 유리
# =========================
scaler_x = StandardScaler()
scaler_y = StandardScaler()

X_train_s = scaler_x.fit_transform(X_train)
X_val_s   = scaler_x.transform(X_val)
X_test_s  = scaler_x.transform(X_test)

y_train_s = scaler_y.fit_transform(y_train)
y_val_s   = scaler_y.transform(y_val)
y_test_s  = scaler_y.transform(y_test)

# =========================
# 4. 모델 정의 (BN+Dropout 포함 MLP)
# =========================
def build_model(input_dim: int, output_dim: int) -> keras.Model:
    inputs = keras.Input(shape=(input_dim,), name="taste")
    x = layers.Dense(256, activation='relu')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.15)(x)

    x = layers.Dense(256, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.15)(x)

    x = layers.Dense(128, activation='relu')(x)
    x = layers.BatchNormalization()(x)

    outputs = layers.Dense(output_dim, activation='linear', name="targets")(x)
    model = keras.Model(inputs, outputs, name="coffee_recipe_net")

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="mse",
        metrics=[keras.metrics.MeanAbsoluteError(name="mae")]
    )
    return model

model = build_model(input_dim=X_train_s.shape[1], output_dim=y_train_s.shape[1])
model.summary()

# =========================
# 5. 콜백 & 학습
# =========================
callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=20, restore_best_weights=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=8, min_lr=1e-5, verbose=1)
]

history = model.fit(
    X_train_s, y_train_s,
    validation_data=(X_val_s, y_val_s),
    epochs=300,
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)

# =========================
# 6. 평가 (역스케일링하여 사람이 해석 가능한 단위로)
# =========================
y_pred_s = model.predict(X_test_s)
y_pred   = scaler_y.inverse_transform(y_pred_s)

mae = mean_absolute_error(y_test, y_pred, multioutput='raw_values')
rmse = np.sqrt(np.mean((y_test.values - y_pred)**2, axis=0))
r2   = r2_score(y_test, y_pred, multioutput='raw_values')

for name, m_mae, m_rmse, m_r2 in zip(target_features, mae, rmse, r2):
    print(f"[{name}] MAE: {m_mae:.3f} | RMSE: {m_rmse:.3f} | R²: {m_r2:.3f}")

# =========================
# 7. 레시피 추천 함수 (딥러닝 모델 버전)
#   - 컵 용량 기준 1잔량 계산
#   - 물리적 제약(클리핑) 예시 포함
# =========================
def recommend_recipe(acidity: float, sweet: float, fruit: float, nutty: float,
                     mouthfeel: float, bitter: float = 0.0, sour: float = 0.0,
                     cup_size_ml: float = 250.0) -> Dict:
    # 입력 → DF → 스케일 → 예측 → 역스케일
    user_input = pd.DataFrame([{
        'Acidity': acidity,
        'Sweet': sweet,
        'Fruit': fruit,
        'Nutty': nutty,
        'Mouthfeel': mouthfeel,
        'Bitter': bitter,
        'Sour': sour
    }], columns=taste_features)

    user_s = scaler_x.transform(user_input)
    pred_s = model.predict(user_s)
    dose, brew_mass, extraction = scaler_y.inverse_transform(pred_s)[0].tolist()

    # (선택) 현실적 범위로 클리핑/보정
    # 에스프레소/브루잉이 섞인 데이터일 수 있으니, 대략적인 안전 범위 예시:
    dose = float(np.clip(dose, 8.0, 30.0))              # g
    brew_mass = float(np.clip(brew_mass, 120.0, 450.0)) # g(≈ml)
    extraction = float(np.clip(extraction, 15.0, 26.0)) # %

    # 잔 수 환산
    cup_count = max(brew_mass / cup_size_ml, 1e-6)
    dose_per_cup = dose / cup_count
    brew_per_cup = brew_mass / cup_count

    return {
        f"1잔({int(cup_size_ml)}ml) 기준": {
            "1잔당 도징량 (g)": round(dose_per_cup, 2),
            "1잔당 추출량 (g)": round(brew_per_cup, 2),
            "1잔당 추출률 (%)": round(extraction, 2)
        },
        "배치 기준(전체)": {
            "총 도징량 (g)": round(dose, 2),
            "총 추출량 (g)": round(brew_mass, 2),
            "추출률 (%)": round(extraction, 2)
        }
    }

# 사용 예시
example = recommend_recipe(acidity=6, sweet=4, fruit=5, nutty=3, mouthfeel=5, bitter=2, sour=1, cup_size_ml=250)
print(example)

# =========================
# 8. 아티팩트 저장 (모델/스케일러)
# =========================
SAVE_DIR = "/mnt/data/coffee_model_artifacts"
os.makedirs(SAVE_DIR, exist_ok=True)

model.save(os.path.join(SAVE_DIR, "coffee_recipe_net.keras"))  # Keras v3 권장 포맷
joblib.dump(scaler_x, os.path.join(SAVE_DIR, "scaler_x.bin"))
joblib.dump(scaler_y, os.path.join(SAVE_DIR, "scaler_y.bin"))

print("저장 완료:", SAVE_DIR)


Model: "coffee_recipe_net"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ taste (InputLayer)              │ (None, 7)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │         2,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │        65,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ targets (Dense)                 │ (None, 3)              │           387 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 103,683 (405.01 KB)

 Trainable params: 102,403 (400.01 KB)

 Non-trainable params: 1,280 (5.00 KB)

Epoch 1/300
75/75 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - loss: 1.9481 - mae: 1.1002 - val_loss: 0.9990 - val_mae: 0.8136 - learning_rate: 0.0010
Epoch 2/300
75/75 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 1.1732 - mae: 0.8676 - val_loss: 0.9552 - val_mae: 0.7963 - learning_rate: 0.0010
Epoch 3/300
75/75 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 1.0708 - mae: 0.8232 - val_loss: 0.9408 - val_mae: 0.7914 - learning_rate: 0.0010
Epoch 4/300
75/75 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 1.0163 - mae: 0.8112 - val_loss: 0.9393 - val_mae: 0.7896 - learning_rate: 0.0010
Epoch 5/300
75/75 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.9581 - mae: 0.7832 - val_loss: 0.9382 - val_mae: 0.7885 - learning_rate: 0.0010
Epoch 6/300
75/75 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.9544 - mae: 0.7841 - val_loss: 0.9341 - val_mae: 0.7835 - learning_rate: 0.0010
Epoch 7/300
75/75 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.9364 - mae: 0.7781 - val_loss: 0.9516 - val_mae: 0.7910 - learning_rate: 0.0010
Epoch 8/300
